# Lab 2: Geoparsing I

<p align="center"><img src="https://raw.githubusercontent.com/explosion/spaCy/master/website/src/images/logo.svg" width="250"/></p>

**Name:** _(fill in your first and last name before submission)_

In this lab session, we'll:
- Learn how to use the [spaCy](https://spacy.io) package to perform named entity recognition (NER)
- Perform geoparsing, i.e., extract place names from the NER results

```text
Text          "Mozart was born in Salzburg in 1756 and later moved to Vienna."
                                        │
                                        ▼
NER           Mozart [PERSON]   Salzburg [GPE]   1756 [DATE]   Vienna [GPE]
              (find all named entities and label their types)
                                        │
                                        ▼
Geoparsing    Salzburg [GPE]   Vienna [GPE]
              (keep only the entities that are place names)
```

We'll work with three kinds of text:
- First, an example text
- Then, text from Wikipedia
- Lastly, text in different languages

During the geoparsing process, we'll also focus on three aspects of spaCy usage:
- Tokenization
- Selection of the trained pipeline
- Result visualization

Below I will provide code along with explanations.

## Part 1: Using spaCy on an Example Text

The example text is about [Vienna](https://en.wikipedia.org/wiki/Vienna). Check out what it is.

In [ ]:
example_text = "Vienna is the capital, most populous city, and one of the nine states of Austria. It is Austria's primate city, with just over two million inhabitants. Its larger metropolitan area has a population of nearly 2.9 million, representing nearly one-third of the country's population. Vienna is the cultural, economic, and political centre of the country, the fifth-largest city by population in the European Union, and the most populous of the cities on the river Danube."

print(example_text)

Next, we import spaCy and load a **trained pipeline**. A trained pipeline is a collection of models that spaCy applies to a text.

Here we use [`en_core_web_sm`](https://spacy.io/models/en#en_core_web_sm), the **small English pipeline**. Its name tells us what it is:
- **en**: English
- **core**: a general-purpose pipeline
- **web**: trained on written web text
- **sm**: small (about 12 MB), which makes it fast to load and run

In [ ]:
import spacy

# Load the small English pipeline
nlp = spacy.load("en_core_web_sm")

Now we pass the example text to the pipeline. The result, `doc`, is the processed text. The first thing spaCy does is **tokenization**, i.e., splitting the text into tokens, so `doc` is a sequence of tokens that we can loop over.

For each token, we can print:
- `token.text`: the token itself

While reading the output, look at how spaCy handles punctuation, as well as "Austria's", "2.9", and "one-third".

In [ ]:
# Process the example text with the pipeline
doc = nlp(example_text)

# Print each token
for token in doc:
    print(token.text)

Next comes **NER**. The pipeline has already found the named entities in the text and stored them in `doc.ents`.

For each entity, we can print:
- `ent.text`: the entity itself
- `ent.label_`: its type, e.g., `GPE` (countries, cities, states)

In [ ]:
# Print each named entity with its label
for ent in doc.ents:
    print(ent.text, ent.label_)

What does each label mean? Check the table of entity types in the [spaCy documentation](https://v2.spacy.io/api/annotation#named-entities) and look up the labels that appear in the output above.

A list of entities is hard to read. spaCy comes with a built-in **visualizer** called [displaCy](https://spacy.io/usage/visualizers), which highlights the named entities and their labels directly in the text.

To use it, we call `displacy.render()` with:
- `doc`: the processed text
- `style="ent"`: the entity visualizer

In [ ]:
from spacy import displacy

# Visualize all named entities in the example text
displacy.render(doc, style="ent")

Finally, **geoparsing**. Not every named entity is a place name, so we filter the NER results and keep only the entities whose label can refer to a place:
- `GPE`: countries, cities, states
- `LOC`: non-GPE locations, mountain ranges, bodies of water
- `FAC`: buildings, airports, highways, bridges
- `ORG`: companies, agencies, institutions
- `NORP`: nationalities or religious or political groups

In [ ]:
# Labels that can refer to a place
place_labels = ["GPE", "LOC", "FAC", "ORG", "NORP"]

# Keep only the entities with one of these labels
for ent in doc.ents:
    if ent.label_ in place_labels:
        print(ent.text, ent.label_)

We can visualize the geoparsing result in the same way. The `options` argument tells displaCy which labels to highlight, so we pass it our `place_labels`.

In [ ]:
# Visualize only the place-related entities
displacy.render(doc, style="ent", options={"ents": place_labels})

## Part 2: Using Wikipedia Data

<p align="center"><img src="https://upload.wikimedia.org/wikipedia/commons/8/80/Wikipedia-logo-v2.svg" width="150"/></p>

Actually, the example data in Part I comes from [Wikipedia](https://www.wikipedia.org/). Therefore, we will next work with more data from it.

Same as Lab 1, here we again will use the `requests` package to get data; but we will use a simpler version of the code.

Before looking at the code, here's what each new piece means and how they link together:
- `url`: the web address of the data we want.
- `headers`: a small note we attach to our request to tell Wikipedia who is asking. Make sure to replace `your_email@example.com` with your own email address, otherwise your request might get blocked.
- `requests.get(url, headers=headers)`: actually sends the request to Wikipedia and stores its reply in `response`.
- `response.json()`: Wikipedia's reply comes back in a structured format called JSON; `.json()` converts it so we can use it directly.
- `["extract"]`: picks out just the summary text from the JSON output.

`response.json()` actually contains many fields. A (shortened) example looks like this:
```json
{
  "title": "Innere Stadt",
  "extract": "The Innere Stadt is the 1st municipal district of Vienna..."
}
```

```text
url, headers
   │
   ▼
requests.get(url, headers=headers)      (send the request)
   │
   ▼
response                                 (Wikipedia's reply)
   │
   ▼
response.json()                          (reply converted to a Python dict)
   │
   ▼
["extract"]                              (pick out just the summary text)
   │
   ▼
wiki_text
```

Below is an example about "Innere Stadt".

In [ ]:
import requests

# Get the "Innere Stadt" page summary from Wikipedia
url = "https://en.wikipedia.org/api/rest_v1/page/summary/Innere_Stadt"
headers = {"User-Agent": "GIR-2026-Lab2/1.0 (your_email@example.com)"}
response = requests.get(url, headers=headers)

wiki_text = response.json()["extract"]

print(wiki_text)

**Task**: Now it's your turn. Using `wiki_text` above, replicate the geoparsing steps from Part 1:
- Process `wiki_text` with the `nlp` pipeline
- Print each named entity together with its label
- Filter the entities to keep only the place names (by reusing `place_labels`)
- Visualize the place names with `displacy`

In [ ]:
# Your code here

## Part 3: Using Different Languages

So far we've only worked with English text and spaCy's English pipeline. But GIR systems often need to handle text in other languages too.

spaCy provides [trained pipelines for many languages](https://spacy.io/usage/models). Here we'll try German, using [`de_core_news_sm`](https://spacy.io/models/de#de_core_news_sm), the small German pipeline.

Let's get a German text to try it on: the German Wikipedia summary for the same "Innere Stadt" page, using the same `requests` approach as Part 2.

**Task**: Now it's your turn to perform the entire geoparsing workflow using the German data and pipeline.

In [ ]:
import requests

# Get the German "Innere Stadt" page summary from Wikipedia
url = "https://de.wikipedia.org/api/rest_v1/page/summary/Innere_Stadt_(Wien)"

# The rest of your code here

In [ ]:
import spacy

# Load the small German pipeline
nlp_de = spacy.load("de_core_news_sm")

# The rest of your code here

## Part 4: Place Names Are Everywhere

Place names don't only show up in text that's directly *about* places. They pop up all over the "place".

For example, Taylor Swift's song ["Welcome to New York"](https://en.wikipedia.org/wiki/Welcome_to_New_York_(song)), from her album [*1989*](https://en.wikipedia.org/wiki/1989_(album)), has a place name right in the title; so is ["Cappuccino"](https://en.wikipedia.org/wiki/Cappuccino).

<p align="center"><img src="https://upload.wikimedia.org/wikipedia/en/f/f6/Taylor_Swift_-_1989.png" style="width:300px; height:300px; object-fit:cover;"/> <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/7/70/Cappuccino_in_original.jpg/330px-Cappuccino_in_original.jpg" style="width:300px; height:300px; object-fit:cover;"/></p>

**Task**: Pick your own text that isn't primarily about geography (a song, a kind of food, or anything you like) and run the geoparsing steps. See what place names show up.

In [ ]:
# Your code here